In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

import importlib

# Add current directory to path to allow imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from graph_structure import Graph
from reward_model import reward

# Parameters
n = 50
p = 0.16  # Increased p to ensure connectedness (~8 neighbors per node)
episodes = 200000
alpha = 0.1   # Learning rate
gamma = 0.0   # Discount factor (0 for immediate reward / bandit setting)
temp = 0.5 # Temperature for Softmax (lower = more greedy, higher = more random)

In [2]:
# Initialize Graph and Q-table
graph = Graph(n, p)
q_table = np.zeros((n, 2)) # Rows: Agents, Cols: Actions (0, 1)

# Check graph stats
degrees = [len(graph.get_adjacency_list(i)) for i in range(n)]
print(f"Average Degree: {np.mean(degrees):.2f}")
print(f"Min Degree: {np.min(degrees)}, Max Degree: {np.max(degrees)}")
print(f"Islands (isolated nodes): {degrees.count(0)}")


# Initial random opinions
current_opinions = np.random.randint(0, 2, n)

history_mean_opinion = []
history_q_diff = []
opinions_history = [current_opinions.copy()]

print(f"Starting simulation with Softmax (T={temp})...")

Average Degree: 9.76
Min Degree: 3, Max Degree: 19
Islands (isolated nodes): 0
Starting simulation with Softmax (T=0.5)...


In [3]:
def softmax(x, temp):
    # Shift values for stability
    e_x = np.exp((x - np.max(x)) / temp)
    return e_x / e_x.sum()


temperature = np.full(n, temp)
epsilon = 0.06
leaders = []
for i in range(1, n):
    if np.random.random() < epsilon:
       temperature[i] = 0.001 # Extremely low temp = strict argmax
       leaders.append(i)

In [4]:
leaders

[4, 12, 19, 36]

In [5]:
for episode in range(episodes):
    new_opinions = np.zeros(n, dtype=int)
    
    # Select actions (Softmax)
    for i in range(n):
        if i in leaders:
            new_opinions[i] = opinions_history[-1][i]
            continue
        probs = softmax(q_table[i], temperature[i])
        new_opinions[i] = np.random.choice([0, 1], p=probs)
    
    # Calculate rewards and update Q-table
    for i in range(n):
        action = new_opinions[i]
        r = reward(2 * action - 1, i, graph, new_opinions)

        # Q-Learning Update
        best_future_q = np.max(q_table[i])
        q_table[i, action] += alpha * (r + gamma * best_future_q - q_table[i, action])
        
    current_opinions = new_opinions
    
    # Metrics
    mean_op = np.mean(current_opinions)
    history_mean_opinion.append(mean_op)
    opinions_history.append(current_opinions.copy())
    
    q_diff = np.mean(np.abs(q_table[:, 1] - q_table[:, 0]))
    history_q_diff.append(q_diff)
    
    if episode % 20 == 0:
        print(f"Episode {episode}: Mean Opinion = {mean_op:.2f}, Mean Q-Diff = {q_diff:.4f}")

print("Simulation complete.")

Episode 0: Mean Opinion = 0.46, Mean Q-Diff = 0.0472
Episode 20: Mean Opinion = 0.64, Mean Q-Diff = 0.1228
Episode 40: Mean Opinion = 0.44, Mean Q-Diff = 0.0928
Episode 60: Mean Opinion = 0.38, Mean Q-Diff = 0.0733
Episode 80: Mean Opinion = 0.42, Mean Q-Diff = 0.0828
Episode 100: Mean Opinion = 0.40, Mean Q-Diff = 0.0783
Episode 120: Mean Opinion = 0.50, Mean Q-Diff = 0.0782
Episode 140: Mean Opinion = 0.50, Mean Q-Diff = 0.0807
Episode 160: Mean Opinion = 0.54, Mean Q-Diff = 0.0815
Episode 180: Mean Opinion = 0.48, Mean Q-Diff = 0.0668
Episode 200: Mean Opinion = 0.52, Mean Q-Diff = 0.0814
Episode 220: Mean Opinion = 0.54, Mean Q-Diff = 0.0841
Episode 160: Mean Opinion = 0.54, Mean Q-Diff = 0.0815
Episode 180: Mean Opinion = 0.48, Mean Q-Diff = 0.0668
Episode 200: Mean Opinion = 0.52, Mean Q-Diff = 0.0814
Episode 220: Mean Opinion = 0.54, Mean Q-Diff = 0.0841
Episode 240: Mean Opinion = 0.44, Mean Q-Diff = 0.0769
Episode 260: Mean Opinion = 0.46, Mean Q-Diff = 0.0832
Episode 280: Mea

In [6]:
# Visualization using the external module
import viz # Reload to ensure latest code is used

viz.plot_metrics(history_mean_opinion, history_q_diff)
viz.plot_distribution(current_opinions)
#viz.save_animation(opinions_history, graph, n, leaders=leaders)

Metrics saved to results/training_metrics.png
Distribution saved to results/final_distribution.png
